# Demo 6: Build your own RL application

You describe the problem, the robot finds the policy with Q-learning. You only edit the problem, the learning itself is done behind the scenes.

| Part | What you do |
|---|---|
| **A. Define** | states, actions, transition probabilities, rewards |
| **B. Set** | learning rate, exploration, risk, discount |
| **C. Train** | train a little and save, then load, train further and save again |
| **D. Change** | change the rewards (or anything else) and let the saved robot relearn |
| **E. Your own** | replace the example with your own problem |

**The example** is deliberately abstract: 3 states (A, B, C) and 3 actions (1, 2, 3). Everything is in two matrices: what happens after each action, and what it earns.

**How to use:** click *Run → Run All Cells* once to see the whole story. Then edit a cell, run it, and run the training cells you need. Training always uses the definitions and settings you ran last.

## Switch on the robot

Run this cell first. The learning, checking and drawing happen behind the scenes, there is nothing to edit here.

In [ ]:
from rl_demos.build_your_own import *

connect(globals())    # lets the robot read your cells

## A. Define the problem

### 1. States and actions

A state must hold everything that matters for what happens next. Here every action is possible in every state.

In [ ]:
STATES = ['A', 'B', 'C']
ACTIONS = [1, 2, 3]

START = {'A': 1.0}     # where an episode starts, with which probability

### 2. Transition probabilities: a 3 × 3 × 3 matrix

`TRANSITIONS[state][action][next state]` = $P(s' \mid s, a)$. One block per state, one line per action, one number per next state, all in the order of `STATES` and `ACTIONS`. Every line adds up to 1.

The robot never sees these numbers. They are the simulated world it practises in.

In [ ]:
TRANSITIONS = [
    #   to:   A    B    C
    [                         # from A
        [0.1, 0.5, 0.4],      #   action 1
        [0.0, 1.0, 0.0],      #   action 2
        [0.5, 0.5, 0.0],      #   action 3
    ],
    [                         # from B
        [0.3, 0.3, 0.4],
        [0.0, 1.0, 0.0],
        [0.0, 0.0, 1.0],
    ],
    [                         # from C
        [0.0, 0.0, 1.0],
        [1.0, 0.0, 0.0],
        [0.5, 0.1, 0.4],
    ],
]

show_diagram()      # draws these probabilities: one picture per action

### 3. Rewards: also a 3 × 3 × 3 matrix

`REWARDS[state][action][next state]`: the reward for taking that action in that state and landing in that next state. The robot does exactly what you reward, which is not always what you meant.

In the example, landing in C is worth 10, and action 2 costs 1 and action 3 costs 5. You can make every number different.

In [ ]:
REWARDS = [
    #   to:  A   B   C
    [                      # from A
        [ 2,  1, 10],      #   action 1
        [ 1,  0,  9],      #   action 2
        [-3, -4,  5],      #   action 3
    ],
    [                      # from B
        [ 2,  1, 10],
        [ 1,  0,  9],
        [-3, -4,  5],
    ],
    [                      # from C
        [ 2,  1, 10],
        [ 1,  0,  9],
        [-3, -4,  5],
    ],
]

show_diagram()      # the same picture, now with the rewards on the arrows

*Optional:* `END_STATES = ['C']` makes C stop the episode, and `LANDING_REWARDS = {'C': 100}` adds a reward for arriving there. Without them, an episode simply lasts `MAX_STEPS` steps.

## B. How the robot learns

| Setting | Low | High |
|---|---|---|
| **Learning rate** | slow but stable | fast but jumpy |
| **Exploration** | may never find something better, slow to notice change | keeps trying things that are known to be bad |
| **Risk** | −1: losses hurt twice as much (cautious) | 1: losses are ignored (reckless) |
| **Discount** | only the next reward counts | the long term counts |

Risk only changes what the robot *feels* while learning. It is always scored on the real rewards.

In [ ]:
LEARNING_RATE = 0.02   # above 0, up to 1
EXPLORATION = 0.2      # 0 to 1: how often it tries a random action
RISK = 0.0             # -1 (cautious) to 1 (reckless), 0 = neutral
DISCOUNT = 0.9         # 0 up to, not including, 1
MAX_STEPS = 50         # steps per practice episode
SEED = 0               # change it to see whether your result was luck

### Check your problem

Run this after every change in part A. It lists mistakes, such as a line that does not add up to 1 or a matrix of the wrong size, and shows both matrices as one table.

In [ ]:
show_problem()

## C. Train, save, train further

### 5. Start with an empty robot

Run this cell again whenever you want it to forget everything.

In [ ]:
robot = new_robot()

### 6. Train up to a point and save

Only a little practice, so expect mistakes. The file holds the policy and the Q-matrix: you need the Q-matrix to train further.

In [ ]:
train(robot, episodes=20, save_as='v1')
show_policy(robot)

### 7. Load it and train further

To go further still, change the names: load `'v2'`, save as `'v2b'`.

In [ ]:
robot = load('v1')
train(robot, episodes=5000, save_as='v2')
show_policy(robot)
show_learning({'robot': robot})

In [ ]:
# Watch the policy at work (run again for another episode)
watch(robot)

In [ ]:
# The policy as a picture: only the arrows of the action the robot chooses
show_diagram(robot)

## D. The world changes: relearn

### 8. Change something

In the example, landing in C becomes worth 7 less. You can also edit and run any cell in part A or B, for example the probabilities or `RISK`.

This cell changes `REWARDS`. To undo it, run the rewards cell (3) again.

In [ ]:
for state in range(3):            # every block
    for action in range(3):       # every line
        REWARDS[state][action][2] -= 7      # the last column: landing in C
print('Landing in C is now worth', REWARDS[0][0][2])

### 9. How good is the saved policy now?

It has not learned anything yet about the change.

In [ ]:
compare({'v2 (saved before the change)': load('v2')})

### 10. Relearn from the saved policy

`v2` stays as it was, the relearned robot is saved as `v3`. In the chart you see the drop at the moment of the change, and the recovery.

In [ ]:
robot = load('v2')
train(robot, episodes=5000, save_as='v3')
show_policy(robot)
show_learning({'robot': robot})

### 11. Was relearning better than starting again?

Old knowledge often gives a head start, but after a big change it can also get in the way.

In [ ]:
fresh = train(new_robot(), episodes=5000)
compare({'v2 old policy': load('v2'), 'v3 relearned': robot, 'started again': fresh})
show_learning({'relearned': robot, 'started again': fresh}, last_run=True)

## E. Now build your own

First answer these five questions on paper:

1. What decision is taken **again and again**? If it is taken once, you do not need RL.
2. What does the robot **know** at that moment? That is the state.
3. What can it **do**? Those are the actions.
4. How does what it does change what happens **next**? Those are the transitions. If it changes nothing, you do not need RL either.
5. What do you really want, and how could a robot **cheat** on your reward?

Then replace cells 1 to 3 with your version, and change or empty the example in cell 8. Give your saves new names. A small template with 2 states and 2 actions:

```python
STATES = ['quiet', 'busy']
ACTIONS = ['wait', 'advertise']
START = {'quiet': 1.0}

TRANSITIONS = [
    #  to: quiet busy
    [[0.9, 0.1],      # quiet, wait
     [0.3, 0.7]],     # quiet, advertise
    [[0.4, 0.6],      # busy, wait
     [0.4, 0.6]],     # busy, advertise
]

REWARDS = [
    #  to: quiet busy
    [[1, 5],          # quiet, wait
     [-2, 3]],        # quiet, advertise
    [[1, 5],          # busy, wait
     [-2, 3]],        # busy, advertise
]
```

If the reward does not depend on where you land, one number per action is enough: `REWARDS = [[1, -2], [5, 3]]`.

## Before you use this for real

- **The simulator is an assumption.** The policy is only as good as the probabilities you typed. In practice you estimate them from data and test how wrong they may be.
- **Exploration costs real money and can harm real people.** Practise in simulation, limit the allowed actions and keep a person in control.
- **Compare with a simple rule** before you trust or pay for RL.
- **Keep monitoring.** When the world changes, a saved policy gets worse without any warning.
- **Tables stop working** when there are too many states. That is the subject of demo 7.
- **Download your files** from `saved_policies` on Binder: they disappear when the session stops.